**Fix the Dtypes**

 Tip: pd.to_datetime() and pd.to_numeric() are your friends. If a numeric conversion fails, look at WHY before reaching for errors='coerce' — know what you're coercing.

In [1]:
import pandas as pandy
import numpy as numpy

In [2]:
prep_df = pandy.read_csv('../Data/citibike_weather_daily.csv')
prep_df['ride_date'] = pandy.to_datetime(prep_df['ride_date']).dt.normalize()
prep_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1610 entries, 0 to 1609
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ride_date         1610 non-null   datetime64[us]
 1   num_rides         1610 non-null   int64         
 2   avg_duration_min  1610 non-null   float64       
 3   temp_f            1610 non-null   float64       
 4   max_temp_f        1610 non-null   float64       
 5   min_temp_f        1610 non-null   float64       
 6   wind_speed_knots  1610 non-null   float64       
 7   precip_in         1610 non-null   float64       
 8   day_of_week       1610 non-null   str           
 9   month             1610 non-null   int64         
dtypes: datetime64[us](1), float64(6), int64(2), str(1)
memory usage: 125.9 KB


**Handle the Coded Missing Values**

Tip: Think about how many rows are affected and what imputation would be reasonable for that variable (e.g., a nearby day's value, a median, or zero — which makes sense for THIS variable?).

In [3]:
# Only one row seemed to have a missing value: precip_in. 
# Replaced with NaN and filled with mean b/c of previous NOAA encoding doctrine

prep_df['precip_in'] = prep_df['precip_in'].replace(99.99, numpy.nan)
prep_df['precip_in'] = prep_df['precip_in'].fillna(prep_df['precip_in'].mean()) 
prep_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 1610 entries, 0 to 1609
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ride_date         1610 non-null   datetime64[us]
 1   num_rides         1610 non-null   int64         
 2   avg_duration_min  1610 non-null   float64       
 3   temp_f            1610 non-null   float64       
 4   max_temp_f        1610 non-null   float64       
 5   min_temp_f        1610 non-null   float64       
 6   wind_speed_knots  1610 non-null   float64       
 7   precip_in         1610 non-null   float64       
 8   day_of_week       1610 non-null   str           
 9   month             1610 non-null   int64         
dtypes: datetime64[us](1), float64(6), int64(2), str(1)
memory usage: 125.9 KB


**Encode Day of Week**

 Tip: pd.get_dummies(). Look up what drop_first=True does and decide whether to use it — either choice is fine if you can say why.

In [4]:
# Dropped day_of_week b/c it won't be used in the model and is no longer needed
# Dummies created for weekdays will be used instead

prep_df.head(5)

,ride_date,num_rides,avg_duration_min,temp_f,max_temp_f,min_temp_f,wind_speed_knots,precip_in,day_of_week,month
0,2013-07-01,16650,16.31,74.8,78.1,73.4,7.8,0.00,Monday,7
1,2013-07-02,22745,15.97,76.1,82.9,73.0,8.0,0.73,Tuesday,7
2,2013-07-03,21864,16.24,78.5,84.9,73.9,8.8,0.06,Wednesday,7
3,2013-07-04,22326,21.22,82.0,91.0,73.9,8.6,0.96,Thursday,7
4,2013-07-05,21842,18.04,84.4,93.0,75.9,9.0,0.00,Friday,7


In [5]:
prep_df = pandy.get_dummies(prep_df, columns = ['day_of_week'], drop_first = True, dtype = int)
# Not sure what happened, but the dummies were created. The steps are out of order
# drop_first = True drops Friday, but tbh I'm still confused as to the logic 
prep_df.columns 

Index(['ride_date', 'num_rides', 'avg_duration_min', 'temp_f', 'max_temp_f',
       'min_temp_f', 'wind_speed_knots', 'precip_in', 'month',
       'day_of_week_Monday', 'day_of_week_Saturday', 'day_of_week_Sunday',
       'day_of_week_Thursday', 'day_of_week_Tuesday', 'day_of_week_Wednesday'],
      dtype='str')

In [7]:
prep_df.columns = prep_df.columns.str.replace('day_of_week_', '', regex=False)

prep_df.head(5)

,ride_date,num_rides,avg_duration_min,temp_f,max_temp_f,min_temp_f,wind_speed_knots,precip_in,month,Monday,Saturday,Sunday,Thursday,Tuesday,Wednesday
0,2013-07-01,16650,16.31,74.8,78.1,73.4,7.8,0.00,7,1,0,0,0,0,0
1,2013-07-02,22745,15.97,76.1,82.9,73.0,8.0,0.73,7,0,0,0,0,1,0
2,2013-07-03,21864,16.24,78.5,84.9,73.9,8.8,0.06,7,0,0,0,0,0,1
3,2013-07-04,22326,21.22,82.0,91.0,73.9,8.6,0.96,7,0,0,0,1,0,0
4,2013-07-05,21842,18.04,84.4,93.0,75.9,9.0,0.00,7,0,0,0,0,0,0


**Build a Trend Feature**

 Tip: If ride_date is a proper datetime, .dt.year is one option; subtracting the first date and taking .dt.days is another.

In [8]:
#Using years as a trending variable

prep_df['year'] = prep_df['ride_date'].dt.year
prep_df.head(5)

,ride_date,num_rides,avg_duration_min,temp_f,max_temp_f,min_temp_f,wind_speed_knots,precip_in,month,Monday,Saturday,Sunday,Thursday,Tuesday,Wednesday,year
0,2013-07-01,16650,16.31,74.8,78.1,73.4,7.8,0.00,7,1,0,0,0,0,0,2013
1,2013-07-02,22745,15.97,76.1,82.9,73.0,8.0,0.73,7,0,0,0,0,1,0,2013
2,2013-07-03,21864,16.24,78.5,84.9,73.9,8.8,0.06,7,0,0,0,0,0,1,2013
3,2013-07-04,22326,21.22,82.0,91.0,73.9,8.6,0.96,7,0,0,0,1,0,0,2013
4,2013-07-05,21842,18.04,84.4,93.0,75.9,9.0,0.00,7,0,0,0,0,0,0,2013


**Engineer Smarter Features - Optional**

 Tip: A squared feature is how a LINEAR model captures a CURVED relationship — the model is still linear in its coefficients.

 For the sake of time, I will not be completing this step right now.

In [9]:
# Optional, for those who want to push the model further.
# Ideas: a squared temperature term (revisit what you saw in E3 on the hottest days); an is_weekend flag; a rained-at-all binary flag; a US-holidays flag. 
# Each one you add, justify with one sentence tying it to something you observed in EDA.

**Save the Clean Dataset**

In [10]:
prep_df.to_csv('../Data/citibike_weather_daily_cleaned.csv', index=False)
